# Netflix Content Strategy — 10 Business SQL Queries

**Dataset:** Netflix Movies & TV Shows (Kaggle, 8,807 titles)
**Engine:** DuckDB (in-memory analytics)

Each query answers a specific strategic question for content acquisition and portfolio management.

In [1]:
import pandas as pd
import duckdb

df = pd.read_csv('../data/netflix_titles.csv')
df['date_added'] = pd.to_datetime(df['date_added'], errors='coerce')

con = duckdb.connect()
con.register('netflix', df)

## Q1: Content Type Distribution (Movies vs TV Shows)

**Business Question:** What is the balance between movies and TV shows in the catalog?

In [2]:
q1 = con.execute("""
    SELECT type, COUNT(*) as total_titles,
           ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 1) as pct
    FROM netflix
    GROUP BY type
    ORDER BY total_titles DESC
""").fetchdf()
print(q1.to_string(index=False))

   type  total_titles  pct
  Movie          6131 69.6
TV Show          2676 30.4


## Q2: Top Content-Producing Countries

**Business Question:** Which countries supply the most content to Netflix?

In [3]:
q2 = con.execute("""
    WITH split_countries AS (
        SELECT TRIM(UNNEST(STRING_SPLIT(country, ','))) as country
        FROM netflix
        WHERE country IS NOT NULL
    )
    SELECT country, COUNT(*) as title_count,
           ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM split_countries), 1) as pct_of_total
    FROM split_countries
    WHERE country != ''
    GROUP BY country
    ORDER BY title_count DESC
    LIMIT 10
""").fetchdf()
print(q2.to_string(index=False))

       country  title_count  pct_of_total
 United States         3690          36.8
         India         1046          10.4
United Kingdom          806           8.0
        Canada          445           4.4
        France          393           3.9
         Japan          318           3.2
         Spain          232           2.3
   South Korea          231           2.3
       Germany          226           2.3
        Mexico          169           1.7


## Q3: Genre Saturation Analysis

**Business Question:** Which genres are oversaturated and which have room for expansion?

In [4]:
q3 = con.execute("""
    WITH split_genres AS (
        SELECT TRIM(UNNEST(STRING_SPLIT(listed_in, ','))) as genre
        FROM netflix
        WHERE listed_in IS NOT NULL
    )
    SELECT genre, COUNT(*) as title_count,
           ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM split_genres), 2) as market_share_pct
    FROM split_genres
    WHERE genre != ''
    GROUP BY genre
    ORDER BY title_count DESC
    LIMIT 15
""").fetchdf()
print(q3.to_string(index=False))

                   genre  title_count  market_share_pct
    International Movies         2752             14.24
                  Dramas         2427             12.56
                Comedies         1674              8.66
  International TV Shows         1351              6.99
           Documentaries          869              4.50
      Action & Adventure          859              4.45
               TV Dramas          763              3.95
      Independent Movies          756              3.91
Children & Family Movies          641              3.32
         Romantic Movies          616              3.19


## Q4: Content Lifecycle (Release Year → Netflix)

**Business Question:** How long does content take from original release to appearing on Netflix?

In [5]:
q4 = con.execute("""
    SELECT type,
           COUNT(*) as titles,
           ROUND(AVG(EXTRACT(YEAR FROM date_added) - release_year), 1) as avg_years_to_netflix,
           ROUND(MEDIAN(EXTRACT(YEAR FROM date_added) - release_year), 1) as median_years,
           MIN(EXTRACT(YEAR FROM date_added) - release_year) as min_gap,
           MAX(EXTRACT(YEAR FROM date_added) - release_year) as max_gap
    FROM netflix
    WHERE date_added IS NOT NULL
      AND release_year IS NOT NULL
      AND (EXTRACT(YEAR FROM date_added) - release_year) BETWEEN 0 AND 50
    GROUP BY type
    ORDER BY avg_years_to_netflix DESC
""").fetchdf()
print(q4.to_string(index=False))

   type  titles  avg_years_to_netflix  median_years  min_gap  max_gap
  Movie    6083                   5.3           2.0        0       50
TV Show    2562                   2.1           0.0        0       46


## Q5: Regional Gap Analysis

**Business Question:** Which countries have below-average content representation relative to the global average?

In [6]:
q5 = con.execute("""
    WITH split_countries AS (
        SELECT TRIM(UNNEST(STRING_SPLIT(country, ','))) as country
        FROM netflix
        WHERE country IS NOT NULL
    ),
    country_stats AS (
        SELECT country, COUNT(*) as title_count
        FROM split_countries
        WHERE country != ''
        GROUP BY country
    ),
    global_avg AS (
        SELECT ROUND(AVG(title_count), 1) as avg_titles_per_country
        FROM country_stats
    )
    SELECT c.country, c.title_count, g.avg_titles_per_country,
           ROUND(c.title_count - g.avg_titles_per_country, 1) as gap_vs_avg
    FROM country_stats c, global_avg g
    WHERE c.title_count < g.avg_titles_per_country
    ORDER BY c.title_count ASC
    LIMIT 20
""").fetchdf()
print(q5.to_string(index=False))

           country  title_count  avg_titles_per_country  gap_vs_avg
         Nicaragua            1                    82.1       -81.1
          Paraguay            1                    82.1       -81.1
        Azerbaijan            1                    82.1       -81.1
          Slovakia            1                    82.1       -81.1
     Liechtenstein            1                    82.1       -81.1
            Malawi            1                    82.1       -81.1
      Vatican City            1                    82.1       -81.1
           Armenia            1                    82.1       -81.1
Dominican Republic            1                    82.1       -81.1
           Ecuador            1                    82.1       -81.1


## Q6: Rating Distribution & Target Audience

**Business Question:** What audience segments does Netflix's content target?

In [7]:
q6 = con.execute("""
    SELECT rating,
           COUNT(*) as titles,
           ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 1) as pct,
           CASE
               WHEN rating IN ('TV-Y', 'TV-Y7', 'TV-Y7-FV', 'G') THEN 'Kids/Family'
               WHEN rating IN ('PG', 'TV-G', 'TV-PG') THEN 'General/All Ages'
               WHEN rating IN ('PG-13', 'TV-14') THEN 'Teens (13+)'
               WHEN rating IN ('R', 'TV-MA', 'NC-17') THEN 'Mature (17+)'
               WHEN rating = 'NR' THEN 'Not Rated'
               ELSE 'Other'
           END as target_audience
    FROM netflix
    WHERE rating IS NOT NULL
    GROUP BY rating
    ORDER BY titles DESC
""").fetchdf()
print(q6.to_string(index=False))

  rating  titles  pct  target_audience
   TV-MA    3207 36.4     Mature (17+)
   TV-14    2160 24.5      Teens (13+)
   TV-PG     863  9.8 General/All Ages
       R     799  9.1     Mature (17+)
   PG-13     490  5.6      Teens (13+)
   TV-Y7     334  3.8      Kids/Family
    TV-Y     307  3.5      Kids/Family
      PG     287  3.3 General/All Ages
    TV-G     220  2.5 General/All Ages
      NR      80  0.9        Not Rated
       G      41  0.5      Kids/Family
TV-Y7-FV       6  0.1      Kids/Family
   NC-17       3  0.0     Mature (17+)
      UR       3  0.0            Other


## Q7: TV Show Season Analysis

**Business Question:** What is the distribution of season counts for TV shows?

In [8]:
q7 = con.execute("""
    SELECT 
        CAST(REPLACE(REPLACE(duration, ' Seasons', ''), ' Season', '') AS INTEGER) as seasons,
        COUNT(*) as show_count,
        ROUND(AVG(release_year), 0) as avg_release_year
    FROM netflix
    WHERE type = 'TV Show'
      AND duration IS NOT NULL
    GROUP BY seasons
    HAVING seasons <= 15
    ORDER BY seasons
""").fetchdf()
print(q7.to_string(index=False))

 seasons  show_count  avg_release_year
       1        1793            2017.0
       2         425            2017.0
       3         199            2018.0
       4          95            2016.0
       5          65            2016.0
       6          33            2014.0
       7          23            2012.0
       8          17            2013.0
       9           9            2013.0
      10           7            2005.0
      11           2            1998.0
      12           2            2018.0
      13           3            2018.0
      15           2            2018.0


## Q8: Content Addition Trends by Year

**Business Question:** How has Netflix's acquisition strategy evolved over time?

In [9]:
q8 = con.execute("""
    SELECT EXTRACT(YEAR FROM date_added) as year_added,
           COUNT(*) as total_added,
           SUM(CASE WHEN type = 'Movie' THEN 1 ELSE 0 END) as movies_added,
           SUM(CASE WHEN type = 'TV Show' THEN 1 ELSE 0 END) as tv_added,
           ROUND(SUM(CASE WHEN type = 'Movie' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1) as movie_pct
    FROM netflix
    WHERE date_added IS NOT NULL
    GROUP BY year_added
    HAVING year_added >= 2008
    ORDER BY year_added DESC
""").fetchdf()
print(q8.to_string(index=False))

 year_added  total_added  movies_added  tv_added  movie_pct
      2021         1498           993       505       66.3
      2020         1878          1284       594       68.4
      2019         1999          1424       575       71.2
      2018         1625          1237       388       76.1
      2017         1164           839       325       72.1
      2016          418           253       165       60.5
      2015            73            56        17       76.7
      2014            23            19         4       82.6
      2013            10             6         4       60.0
      2012             3             3         0      100.0
      2011            13            13         0      100.0
      2010             1             1         0      100.0
      2009             2             2         0      100.0
      2008             2             1         1       50.0


## Q9: Director Concentration Analysis

**Business Question:** Which directors have the largest portfolio presence on Netflix?

In [10]:
q9 = con.execute("""
    WITH split_directors AS (
        SELECT TRIM(UNNEST(STRING_SPLIT(director, ','))) as director
        FROM netflix
        WHERE director IS NOT NULL
    )
    SELECT director, COUNT(*) as title_count,
           ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM split_directors), 2) as portfolio_share_pct
    FROM split_directors
    WHERE director != ''
    GROUP BY director
    ORDER BY title_count DESC
    LIMIT 15
""").fetchdf()
print(q9.to_string(index=False))

           director  title_count  portfolio_share_pct
      Rajiv Chilaka           22                 0.32
          Jan Suter           21                 0.30
        Raúl Campos           19                 0.27
       Marcus Raboy           16                 0.23
        Suhas Kadav           16                 0.23
          Jay Karas           15                 0.21
Cathy Garcia-Molina           13                 0.19
    Martin Scorsese           12                 0.17
    Youssef Chahine           12                 0.17
        Jay Chapman           12                 0.17


## Q10: Emerging Market Content Opportunity Scoring

**Business Question:** Which emerging markets represent the highest opportunity for content investment?

**Scoring Formula:**
- 30% weight on total titles (volume signal)
- 50% weight on gap vs global average (underserved signal)
- 20% weight on TV content count (future growth signal)

Excludes established markets: US, UK, Canada, France, Germany, Japan, South Korea.

In [11]:
q10 = con.execute("""
    WITH split_countries AS (
        SELECT TRIM(UNNEST(STRING_SPLIT(country, ','))) as country, type
        FROM netflix
        WHERE country IS NOT NULL
    ),
    country_totals AS (
        SELECT country,
               COUNT(*) as total_titles,
               SUM(CASE WHEN type = 'TV Show' THEN 1 ELSE 0 END) as tv_count,
               SUM(CASE WHEN type = 'Movie' THEN 1 ELSE 0 END) as movie_count
        FROM split_countries
        WHERE country != ''
        GROUP BY country
    ),
    global_avg AS (
        SELECT AVG(total_titles) as avg_titles
        FROM country_totals
    )
    SELECT country,
           total_titles,
           tv_count,
           movie_count,
           ROUND(tv_count * 100.0 / total_titles, 1) as tv_pct,
           ROUND(total_titles - (SELECT avg_titles FROM global_avg), 1) as gap_vs_avg,
           ROUND(
               (total_titles * 0.3) + 
               ((total_titles - (SELECT avg_titles FROM global_avg)) * 0.5) + 
               (tv_count * 0.2),
               1
           ) as opportunity_score
    FROM country_totals
    WHERE total_titles < (SELECT avg_titles * 2 FROM global_avg)
      AND total_titles >= 5
      AND country NOT IN ('United States', 'United Kingdom', 'Canada', 'France', 'Germany', 'Japan', 'South Korea')
    ORDER BY opportunity_score DESC
    LIMIT 15
""").fetchdf()
print(q10.to_string(index=False))

     country  total_titles  tv_count  movie_count  tv_pct  gap_vs_avg  opportunity_score
   Australia           160      66.0         94.0    41.3        77.9              100.2
       China           162      48.0        114.0    29.6        79.9               98.2
       Egypt           117      15.0        102.0    12.8        34.9               55.6
      Turkey           113      30.0         83.0    26.5        30.9               55.4
      Taiwan            89      70.0         19.0    78.7         6.9               44.2
   Hong Kong           105       5.0        100.0     4.8        22.9               44.0
       Italy           100      25.0         75.0    25.0        17.9               44.0
     Nigeria           103       9.0         94.0     8.7        20.9               43.2
      Brazil            97      31.0         66.0    32.0        14.9               42.8
   Argentina            91      20.0         71.0    22.0         8.9               35.8
     Belgium         